In [23]:
from pathlib import Path
import pandas as pd
from cyvcf2 import VCF
from tqdm import tqdm

In [24]:
snp_only_dir = Path("./snp_only")

vcf_files = sorted(snp_only_dir.glob("*.vcf"))

In [25]:
# Collect all unique SNP sites across samples 
all_sites = set()
sample_sites = {}

print("🔍 Reading SNP sites from all VCFs...")
for vcf_file in tqdm(vcf_files):
    sample_name = vcf_file.stem.replace("_snps_only", "")
    reader = VCF(str(vcf_file))
    sites = []
    for record in reader:
        # Use (CHROM, POS, REF, ALT) tuple for uniqueness
        for alt in record.ALT:
            site = (record.CHROM, record.POS, record.REF, str(alt))
            sites.append(site)
            all_sites.add(site)
    sample_sites[sample_name] = sites

🔍 Reading SNP sites from all VCFs...


100%|██████████| 192/192 [00:07<00:00, 26.63it/s]


In [26]:
all_sites = sorted(all_sites)

print(f"✅ Total unique SNP sites: {len(all_sites)}")

✅ Total unique SNP sites: 10496


In [27]:
matrix = []
for sample_name, sites in tqdm(sample_sites.items(), desc="🧬 Building SNP matrix"):
    sites_set = set(sites)
    row = [1 if site in sites_set else 0 for site in all_sites]
    matrix.append(row)

🧬 Building SNP matrix:   0%|          | 0/192 [00:00<?, ?it/s]

🧬 Building SNP matrix: 100%|██████████| 192/192 [00:00<00:00, 390.32it/s]


In [28]:
# siguro next time, paki-add yung country and all huhu kung need
columns = [f"{chrom}_{pos}_{ref}_{alt}" for chrom, pos, ref, alt in all_sites]
df = pd.DataFrame(matrix, columns=columns, index=sample_sites.keys())

print("✅ SNP Matrix shape:", df.shape)

✅ SNP Matrix shape: (192, 10496)


In [29]:
# display DataFrame
display(df)

,Chromosome_16_G_T,Chromosome_42_C_T,Chromosome_43_G_T,Chromosome_45_G_T,Chromosome_60_T_A,Chromosome_71_C_T,Chromosome_82_G_C,Chromosome_109_A_C,Chromosome_117_G_A,Chromosome_135_G_A,...,Chromosome_4408213_G_A,Chromosome_4408221_G_T,Chromosome_4408230_G_T,Chromosome_4408273_G_C,Chromosome_4408283_T_C,Chromosome_4408329_G_C,Chromosome_4408423_C_G,Chromosome_4408430_C_T,Chromosome_4408456_G_C,Chromosome_4408479_G_A
ERR046840,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
ERR067667,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ERR067745,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ERR137227,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ERR2184331,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR6831771,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SRR6832095,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SRR6832148,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SRR6964516,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
df.to_csv("snp_matrix.csv")
print("📁 Saved SNP matrix to snp_matrix.csv")

📁 Saved SNP matrix to snp_matrix.csv


In [31]:
# test ko lang if keri na to input sa ml model ganern
check = pd.read_csv('/mnt/c/Users/Lenovo/Downloads/ths-st1-tb/snp_matrix.csv')
check

,Unnamed: 0,Chromosome_16_G_T,Chromosome_42_C_T,Chromosome_43_G_T,Chromosome_45_G_T,Chromosome_60_T_A,Chromosome_71_C_T,Chromosome_82_G_C,Chromosome_109_A_C,Chromosome_117_G_A,...,Chromosome_4408213_G_A,Chromosome_4408221_G_T,Chromosome_4408230_G_T,Chromosome_4408273_G_C,Chromosome_4408283_T_C,Chromosome_4408329_G_C,Chromosome_4408423_C_G,Chromosome_4408430_C_T,Chromosome_4408456_G_C,Chromosome_4408479_G_A
0,ERR046840,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,ERR067667,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ERR067745,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,ERR137227,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ERR2184331,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,SRR6831771,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
188,SRR6832095,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
189,SRR6832148,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
190,SRR6964516,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
